# 🏴󠁧󠁢󠁥󠁮󠁧󠁿 Premier League Opening Weekend Evaluation (V2 Model)

Since our V3 model is currently an architectural shell trained on mock data, we are going to evaluate the newly played Premier League games using our production **V2 Model**.

Because V2 was built for the World Cup and relies on FIFA Rankings, we have to supply a **"Pseudo-FIFA Rank"** for the 20 Premier League clubs so the model understands who the favorites and underdogs are. We have mapped Man City to 1, Arsenal to 2, down to the newly promoted sides at 18-20.

In [2]:
import sys
import urllib.request
import ssl
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Import common for V2 logic
sys.path.append(str(Path.cwd().parent))
import common as c

plt.style.use('ggplot')


## 1. Club Ranks & API Setup

In [3]:
# Pseudo-FIFA Rankings for the 24/25 Premier League based on pre-season odds
PL_RANKS = {
    "Manchester City": 1,
    "Arsenal": 2,
    "Liverpool": 3,
    "Chelsea": 4,
    "Tottenham Hotspur": 5,
    "Manchester United": 6,
    "Newcastle United": 7,
    "Aston Villa": 8,
    "Brighton": 9,
    "West Ham United": 10,
    "Crystal Palace": 11,
    "Bournemouth": 12,
    "Fulham": 13,
    "Wolverhampton": 14,
    "Everton": 15,
    "Brentford": 16,
    "Nottingham Forest": 17,
    "Leicester City": 18,
    "Southampton": 19,
    "Ipswich Town": 20
}

# Inject these into V2's ranking system so it understands the clubs
c.FIFA_RANK.update(PL_RANKS)

import os
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")
API_KEY = os.getenv("SOFASCORE_API_KEY")
HOST = "sofascore.p.rapidapi.com"

def fetch(endpoint):
    url = f"https://{HOST}/{endpoint}"
    req = urllib.request.Request(url, headers={'x-rapidapi-key': API_KEY, 'x-rapidapi-host': HOST})
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    try:
        with urllib.request.urlopen(req, context=ctx) as res:
            if res.status == 200:
                return json.loads(res.read().decode())
    except Exception as e:
        print(f"Error fetching {endpoint}: {e}")
    return None


## 2. Fetch Opening Weekend Matches
We pull the results of the recent Premier League games (Tournament ID 17).

In [4]:
# Fetch recent events for Premier League (ID 17)
# Note: 61627 is the 24/25 season ID for the Premier League in Sofascore
season_data = fetch("tournaments/get-events?tournamentId=17&seasonId=61627&page=0")

pl_matches = []
if season_data and 'events' in season_data:
    # Filter for matches that are finished ('closed')
    pl_matches = [e for e in season_data['events'] if e.get('status', {}).get('type') == 'finished']
    # Sort by start timestamp descending (most recent first)
    pl_matches = sorted(pl_matches, key=lambda x: x.get('startTimestamp', 0), reverse=True)
    
    # Let's just grab the most recent 10 (which is exactly Matchweek 1)
    pl_matches = pl_matches[:10]
    print(f"Found {len(pl_matches)} recent Premier League matches!")
else:
    print("Could not fetch matches. Check API limits.")


Error fetching tournaments/get-events?tournamentId=17&seasonId=61627&page=0: HTTP Error 404: Not Found
Could not fetch matches. Check API limits.


## 3. Predict & Evaluate with V2
For each match, we'll check what the model predicted at Minute 0 (Kickoff) based solely on the pseudo-ranks, and compare it to the actual final result.

In [5]:
model, scaler, T = c.load_model()
results = []

for match in pl_matches:
    home = match['homeTeam']['name']
    away = match['awayTeam']['name']
    hs = match['homeScore'].get('current', 0)
    as_ = match['awayScore'].get('current', 0)
    
    # Clean names to match our PL_RANKS
    # Sofascore sometimes returns slightly different strings, we might need to map them
    home_clean = "Manchester United" if home == "Manchester Utd" else home
    away_clean = "Manchester United" if away == "Manchester Utd" else away
    
    # Generate Pre-Game Feature Row (Minute 0, 0-0, No Lead Changes)
    # c.build_feature_row(home_code, away_code, minute, hs, as_, lead_changes, goals_so_far, is_knockout)
    feat_row = c.build_feature_row(home_clean, away_clean, 0, 0, 0, 0, 0, 0)
    p_away, p_draw, p_home = c.predict(model, scaler, T, feat_row)
    
    actual = "Home" if hs > as_ else ("Away" if as_ > hs else "Draw")
    predicted = "Home" if p_home > p_away and p_home > p_draw else ("Away" if p_away > p_home and p_away > p_draw else "Draw")
    
    results.append({
        "Match": f"{home} {hs} - {as_} {away}",
        "P(Home)": f"{p_home*100:.1f}%",
        "P(Draw)": f"{p_draw*100:.1f}%",
        "P(Away)": f"{p_away*100:.1f}%",
        "Actual": actual,
        "Predicted": predicted,
        "Correct": actual == predicted
    })

df_res = pd.DataFrame(results)
display(df_res)

acc = df_res['Correct'].sum() / len(df_res) if len(df_res) > 0 else 0
print(f"\nOpening Weekend Pre-Match Accuracy (V2 Model): {acc*100:.1f}%")


2026-09-01 14:51:15.644 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


""



Opening Weekend Pre-Match Accuracy (V2 Model): 0.0%
